# 한국 기업 분기별 매출 시계열 예측 — `Korea_revenue_forecast_v2`

DataGuide 원천(`korea_fs_data_from_DG`)에서 분기 매출액(`item_code='M000904001'`)을 조회해
**SARIMA · ETS · Theta + 앙상블**로 예측하고, 결과를 `korea_revenue_forecast_result`에 저장합니다.

이 노트북은 업로드된 `Korea_revenue_forecast_.py`의 로직을 **그대로 준용**하되, 유지·관리와
타당성 검증이 쉽도록 셀 단위로 재구성한 버전입니다.

### 노트북 구성
1. **환경 설정** — import · 경로 · 모듈 · DB
2. **데이터 조회 / 예측 / 저장 함수** — 원본 `.py`와 동일한 파이프라인
3. **🔍 개별 종목 점검** `inspect_ticker()` — 입력 매출 `tail(20)`와 모델별 예측값 조회 *(타당성 확인용, DB 저장 안 함)*
4. **일괄 예측 실행** — 배치 저장 방식으로 다수 종목 처리

> **저장 방식 결정 (요구사항 4):** ticker 1개의 예측 결과는 약 32행(8분기 × 4모델)에 불과해
> 버퍼링 자체의 메모리 부담은 거의 없습니다. 실제 메모리 압박은 모델 적합 과정에서 발생하며
> 이는 함수 반환 시 해제됩니다. 따라서 **종목별 즉시 저장 대신 `BATCH_SIZE`(기본 50)개를 모아
> 한 번에 저장**해 DB 트랜잭션 횟수를 줄이는 방식을 택했고, 모델 적합 잔여 메모리는
> `GC_EVERY`개마다 `gc.collect()`로 정리합니다.

## 1. 환경 설정

In [1]:
import sys
import os
import gc
import numpy as np
import pandas as pd
import pymysql
from datetime import datetime
from pathlib import Path
from typing import Union, List, Tuple, Dict, Optional
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')

### 1-1. 데이터 소스 설정
필요 시 이 셀의 상수만 수정하면 됩니다.

In [2]:
# ==================== 데이터 소스 설정 (필요 시 여기만 수정) ====================
# 매출 데이터를 조회할 원천 테이블
FS_TABLE = "korea_fs_data_from_DG"

# DataGuide에서 매출액을 가리키는 item_code
#   M000904001 = '매출액(천원)' (IS 시트 원본)
REVENUE_ITEM_CODE = "M000904001"

# ticker 형식: True면 'A005930' (A 접두사 포함), False면 '005930'
TICKER_HAS_A_PREFIX = True


def normalize_ticker_for_dg(ticker) -> str:
    """
    DG 테이블 조회용 ticker 정규화.
    입력이 '005930', 5930, 'A005930' 어떤 형태든
    DG 스키마에 맞게 'A005930' 형태로 통일한다.
    """
    if isinstance(ticker, int):
        raw = f"{ticker:06d}"
    else:
        raw = str(ticker).strip()

    # 'A' 접두사 제거 (이미 있으면) 후 6자리 zfill
    body = raw[1:] if raw.upper().startswith('A') else raw
    body = body.zfill(6)

    return f"A{body}" if TICKER_HAS_A_PREFIX else body

### 1-2. 경로 설정
`DATA` 폴더를 자동 탐색해 `sys.path`에 추가합니다 (랩탑/데스크탑 등 PC 무관).

In [3]:
def setup_universal_paths():
    """
    어떤 PC에서도 작동하는 범용 경로 설정 v2 — DATA 폴더 자동 탐색.
    탐색 순서:
      ① 환경변수 DATA_DIR (직접 지정 시)
      ② 현재 폴더와 상위 폴더들의 직속 DATA
      ③ 현재 폴더 하위(깊이 3) 및 상위 3단계 폴더들의 하위(깊이 2)
    old_files/backup 등 백업 폴더 내의 DATA 는 무시한다.
    이 노트북은 대만 프로젝트(config.py)와 무관하며 DATA 폴더만 있으면 동작한다.
    """
    import os
    SKIP = {".git", ".idea", ".venv", "venv", "env", "__pycache__",
            "node_modules", "site-packages", ".ipynb_checkpoints",
            "AppData", "anaconda3", "Miniconda3",
            "old_files", "old", "_old", "backup", "backups", "archive", "bak"}
    current = Path.cwd()

    def register(data_folder):
        parent = data_folder.parent
        for p in (str(parent), str(data_folder)):
            if p not in sys.path:
                sys.path.insert(0, p)
        print("=" * 80)
        print("경로 설정 완료")
        print("=" * 80)
        print(f"프로젝트 루트: {parent}")
        print(f"DATA 폴더:    {data_folder}")
        print(f"현재 위치:     {current}")
        print("=" * 80 + "\n")
        return {"project_root": parent, "data_folder": data_folder, "current": current}

    # ① 환경변수
    env = os.environ.get("DATA_DIR")
    if env and Path(env).exists():
        return register(Path(env))

    # ② 상위 방향: 직속 DATA
    for parent in [current, *current.parents]:
        d = parent / "DATA"
        if d.exists():
            return register(d)

    # ③ 하위 방향: cwd 깊이 3, 상위 3단계는 깊이 2
    def scan(root, max_depth):
        root = Path(root); base = len(root.parts)
        for dp, dn, _ in os.walk(root):
            if len(Path(dp).parts) - base >= max_depth:
                dn[:] = []
            dn[:] = [x for x in dn if x not in SKIP and not x.startswith(".")]
            if "DATA" in dn:
                return Path(dp) / "DATA"
        return None
    hit = scan(current, 3)
    if hit is None:
        for parent in list(current.parents)[:3]:
            hit = scan(parent, 2)
            if hit:
                break
    if hit:
        return register(hit)

    raise FileNotFoundError(
        f"DATA 폴더를 찾을 수 없습니다.\n현재 위치: {current}\n"
        "아래처럼 직접 지정한 뒤 이 셀을 다시 실행하세요:\n"
        '  import os; os.environ["DATA_DIR"] = r"C:\\...\\DATA"'
    )


# 경로 설정 실행
paths = setup_universal_paths()

경로 설정 완료
프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
DATA 폴더:    C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
현재 위치:     C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Korea_Market\analysis\한국기업_매출예측



### 1-3. 예측 모듈 import

> **⚠️ 모듈 이름 확인:** 아래는 업로드한 `.py`와 동일하게 `universal_ts_forecast_function`에서
> import합니다. 만약 DATA 폴더의 실제 파일명이 `universal_ts_forecast_function_v2.py`라면
> 아래 `from universal_ts_forecast_function import ...`를
> `from universal_ts_forecast_function_v2 import ...`로 바꿔주세요. (이 파일은 수정하지 않습니다.)

In [4]:
from universal_ts_forecast_function_v2 import (
    forecast_sarima,
    forecast_ets,
    forecast_theta,
)
from stock_invest_function import get_db_host

print("예측 모듈 import 성공")

예측 모듈 import 성공


### 1-4. DB 연결 설정

In [5]:
def get_db_info() -> dict:
    """DB 연결 정보 반환"""
    return {
        "host": get_db_host(),
        "port": 3307,
        "user": "stox7412",
        "password": "Apt106503!~",
        "database": "investar",
    }


def get_connection(db_info: dict):
    """DB 연결 생성 (pymysql)"""
    return pymysql.connect(
        host=db_info["host"],
        port=int(db_info["port"]),
        user=db_info["user"],
        password=db_info["password"],
        db=db_info["database"],
        charset="utf8mb4",
    )

## 2. 데이터 조회 함수

In [6]:
def get_all_tickers(db_info: dict) -> List[str]:
    """
    DataGuide 테이블에서 매출 데이터가 존재하는 모든 ticker 조회.
    매출액(item_code) 레코드가 하나라도 있는 기업만 대상.
    """
    conn = get_connection(db_info)
    try:
        sql = f"""
              SELECT DISTINCT ticker
              FROM {FS_TABLE}
              WHERE item_code = %s
              ORDER BY ticker
              """
        df = pd.read_sql(sql, conn, params=[REVENUE_ITEM_CODE])
        tickers = df['ticker'].tolist()
        print(f"매출 데이터 보유 ticker {len(tickers)}개 조회 완료 (소스: {FS_TABLE})\n")
        return tickers
    finally:
        conn.close()


def get_quarterly_revenue_simple(
        db_info: dict,
        ticker: Union[str, int],
        adjust_q4: bool = False,
        verbose: bool = False
) -> pd.DataFrame:
    """
    특정 ticker의 분기별 매출 시계열을 DataGuide 테이블에서 조회.
    DataGuide는 각 분기가 독립값이므로 FY 역산이 불필요하다.

    Returns columns:
        report_date, thstrm_amount, bsns_year, quarter, ticker
    """
    ticker_str = normalize_ticker_for_dg(ticker)
    conn = get_connection(db_info)
    try:
        sql = f"""
              SELECT
                  date                  AS report_date,
                  value                 AS thstrm_amount,
                  YEAR(date)            AS bsns_year,
                  ticker                AS ticker
              FROM {FS_TABLE}
              WHERE ticker = %s
                AND item_code = %s
                AND value IS NOT NULL
              ORDER BY date
              """
        df = pd.read_sql(sql, conn, params=[ticker_str, REVENUE_ITEM_CODE])

        if df.empty:
            if verbose:
                print(f"  {ticker_str}: 데이터 없음")
            return df

        # quarter 컬럼 유도 (downstream 호환)
        df['quarter'] = 'Q' + ((pd.to_datetime(df['report_date']).dt.month + 2) // 3).astype(str)

        if verbose:
            print(f"  {ticker_str}: 매출 데이터 {len(df)}행 조회")

        if adjust_q4 and verbose:
            print(f"  [INFO] DG 데이터는 분기 독립값이므로 adjust_fy_to_q4 skip")

        return df
    finally:
        conn.close()

<details>
<summary>📦 [LEGACY] <code>adjust_fy_to_q4()</code> — DART 데이터용 (현재 미사용, 보존용)</summary>

DataGuide 경로에서는 호출하지 않습니다. 과거 DART(누적 공시) 경로로 돌아갈 경우를 대비해 보존합니다.
</details>

In [7]:
def adjust_fy_to_q4(df: pd.DataFrame) -> pd.DataFrame:
    """
    [LEGACY] DART 데이터용. DataGuide 경로에서는 사용하지 않음.
    각 연도별로 FY - (Q1 + Q2 + Q3) = 순수 Q4 로 역산.
    """
    result_df = df.copy()
    for year in result_df['bsns_year'].unique():
        year_mask = result_df['bsns_year'] == year
        fy_mask = year_mask & (result_df['quarter'] == 'FY')
        if fy_mask.any():
            fy_amount = result_df.loc[fy_mask, 'thstrm_amount'].iloc[0]
            q123_mask = year_mask & result_df['quarter'].isin(['Q1', 'H1', 'Q3'])
            q123_sum = result_df.loc[q123_mask, 'thstrm_amount'].sum()
            q4_amount = fy_amount - q123_sum
            result_df.loc[fy_mask, 'thstrm_amount'] = q4_amount
            result_df.loc[fy_mask, 'quarter'] = 'Q4'
    return result_df

## 3. 예측 함수

`_prepare_series()`로 매출 시계열(PeriodIndex)을 만든 뒤 세 모델을 적합합니다.
점검용 `inspect_ticker()`와 일괄용 `forecast_quarterly_revenue()`가 **동일한 전처리·모델 호출**을
공유하도록 분리해, 점검 시 보이는 예측값과 실제 저장값이 어긋나지 않게 했습니다.

In [8]:
def _prepare_series(revenue_df: pd.DataFrame) -> pd.Series:
    """
    조회된 매출 DataFrame을 분기 PeriodIndex 매출 시계열(float)로 변환.
    중복 분기는 마지막 값 사용.
    """
    df = revenue_df[['report_date', 'thstrm_amount']].copy()
    df.columns = ['date', 'revenue']
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['quarter'] = df['date'].dt.quarter
    df['period'] = df['year'].astype(str) + 'Q' + df['quarter'].astype(str)
    df['period'] = pd.PeriodIndex(df['period'], freq='Q')

    # 중복 분기 제거 (각 분기의 마지막 값)
    if df.duplicated(subset=['period'], keep=False).any():
        df = df.sort_values(['period', 'date']).groupby('period').last().reset_index()

    df = df.set_index('period').sort_index()
    return df['revenue'].astype(float)


def _run_models(series: pd.Series, forecast_quarters: int) -> Tuple[pd.DataFrame, dict]:
    """
    SARIMA / ETS / Theta 적합 후 예측 분기 인덱스를 가진 wide DataFrame과
    각 모델의 원본 result dict를 반환.
    """
    m = 4  # 분기 데이터의 계절성

    sarima_result = forecast_sarima(
        y=series, forecast_horizon=forecast_quarters,
        seasonal_period=m, try_transforms=True,
    )
    ets_result = forecast_ets(
        y=series, forecast_horizon=forecast_quarters,
        m=m, try_transforms=True,
    )
    theta_result = forecast_theta(
        y=series, forecast_horizon=forecast_quarters,
        m=m, try_transforms=True,
    )

    last_period = series.index[-1]
    forecast_periods = pd.period_range(start=last_period + 1, periods=forecast_quarters, freq='Q')

    fc = pd.DataFrame({
        "SARIMA": sarima_result.get("forecast"),
        "ETS": ets_result.get("forecast"),
        "Theta": theta_result.get("forecast"),
    }, index=forecast_periods)
    fc["Ensemble"] = fc[["SARIMA", "ETS", "Theta"]].mean(axis=1)

    raw = {"SARIMA": sarima_result, "ETS": ets_result, "Theta": theta_result}
    return fc, raw


def forecast_quarterly_revenue(
        ticker: str,
        db_info: dict,
        forecast_quarters: int = 8,
        min_data_periods: int = 24,
        verbose: bool = False
) -> Tuple[bool, pd.DataFrame, str]:
    """
    단일 ticker 분기 매출 예측.
    Returns: (성공여부, wide 예측 DataFrame[date,SARIMA,ETS,Theta,Ensemble,ticker], 에러메시지)
    """
    try:
        revenue_df = get_quarterly_revenue_simple(db_info, ticker, adjust_q4=False, verbose=verbose)
        if revenue_df.empty:
            return False, pd.DataFrame(), "데이터 없음"
        if len(revenue_df) < min_data_periods:
            return False, pd.DataFrame(), f"데이터 부족 ({len(revenue_df)}개 < {min_data_periods}개)"

        series = _prepare_series(revenue_df)
        fc, _ = _run_models(series, forecast_quarters)

        fc['ticker'] = ticker
        # DatetimeIndex로 변환 (분기 말일)
        fc.index = fc.index.to_timestamp(how='end')
        fc = fc.reset_index()
        fc.columns = ['date'] + list(fc.columns[1:])
        return True, fc, ""

    except Exception as e:
        error_msg = str(e)
        if verbose:
            print(f"  {ticker}: 예측 실패 - {error_msg}")
        return False, pd.DataFrame(), error_msg

## 4. 🔍 개별 종목 점검 — `inspect_ticker()`  *(요구사항 3)*

종목 코드 하나를 넣으면 **예측 입력으로 쓰인 매출 `tail(20)`** 과 **모델별 예측값**(SARIMA·ETS·Theta·Ensemble)을
표로 보여줍니다. **DB에 저장하지 않으므로** 예측 타당성을 자유롭게 확인할 수 있습니다.

반환값(dict)으로 `input_series`, `input_tail`, `forecast`, `raw_results`를 받아 추가 분석도 가능합니다.

In [9]:
def inspect_ticker(
        ticker: Union[str, int],
        db_info: dict = None,
        forecast_quarters: int = 8,
        tail_n: int = 20,
        show: bool = True
) -> dict:
    """
    개별 종목의 예측 입력값과 모델별 예측값을 조회 (타당성 검증용, DB 저장 X).

    Parameters
    ----------
    ticker : 종목 코드 ('005930', 5930, 'A005930' 모두 허용)
    db_info : DB 연결 정보 (None이면 get_db_info() 자동 호출)
    forecast_quarters : 예측 분기 수
    tail_n : 입력 매출 시계열에서 보여줄 최근 분기 수 (기본 20)
    show : True면 화면에 출력

    Returns
    -------
    dict: {ticker, n_obs, input_series, input_tail, forecast, raw_results}
    """
    if db_info is None:
        db_info = get_db_info()

    ticker_str = normalize_ticker_for_dg(ticker)
    revenue_df = get_quarterly_revenue_simple(db_info, ticker, adjust_q4=False, verbose=False)

    if revenue_df.empty:
        print(f"[{ticker_str}] 매출 데이터가 없습니다.")
        return {"ticker": ticker_str, "n_obs": 0, "input_series": None,
                "input_tail": None, "forecast": None, "raw_results": None}

    series = _prepare_series(revenue_df)
    n_obs = len(series)

    # 입력 매출 tail(N) — 보기 좋게 정리
    tail = series.tail(tail_n).rename("revenue(천원)").to_frame()
    tail.index = tail.index.astype(str)  # '2024Q1' 형태

    # 모델별 예측
    fc, raw = _run_models(series, forecast_quarters)
    fc_display = fc.copy()
    fc_display.index = fc_display.index.astype(str)  # 예측 분기 '2025Q1' 형태

    if show:
        print("=" * 70)
        print(f" 종목: {ticker_str}   |   총 관측 분기: {n_obs}개   |   예측 분기: {forecast_quarters}개")
        print("=" * 70)
        print(f"\n[ 예측 입력 매출 — 최근 {min(tail_n, n_obs)}개 분기 (tail) ]")
        print(tail.to_string(float_format=lambda x: f"{x:,.0f}"))

        # SARIMA 파라미터가 있으면 참고용으로 표시
        sarima_order = raw.get("SARIMA", {}).get("order") or raw.get("SARIMA", {}).get("params")
        if sarima_order is not None:
            print(f"\n[ SARIMA 선택 파라미터 ] {sarima_order}")

        print(f"\n[ 모델별 예측값 ]")
        print(fc_display.to_string(float_format=lambda x: f"{x:,.0f}"))
        print("=" * 70)

    return {
        "ticker": ticker_str,
        "n_obs": n_obs,
        "input_series": series,
        "input_tail": tail,
        "forecast": fc,
        "raw_results": raw,
    }

## 5. Long 변환 · 배치 저장 함수

In [10]:
def convert_to_long_format(df: pd.DataFrame) -> pd.DataFrame:
    """
    Wide(date,ticker,SARIMA,ETS,Theta,Ensemble) → Long(date,ticker,indicator,value)
    """
    if df.empty:
        return pd.DataFrame(columns=['date', 'ticker', 'indicator', 'value'])

    model_cols = ['SARIMA', 'ETS', 'Theta', 'Ensemble']
    long_df = df.melt(
        id_vars=['date', 'ticker'],
        value_vars=model_cols,
        var_name='indicator',
        value_name='value'
    )
    return long_df.sort_values(['ticker', 'date', 'indicator']).reset_index(drop=True)


def save_forecasts_batch(
        forecasts_long: pd.DataFrame,
        db_info: dict,
        table_name: str = "korea_revenue_forecast_result",
        batch_size: int = 50
) -> Tuple[int, int, int]:
    """
    예측 결과(Long)를 배치로 DB에 저장. created_at(실행 날짜)으로 버전 구분.
    UNIQUE KEY (date, ticker, indicator, created_at) → 같은 날 재실행 시 value 갱신.

    Returns: (신규 삽입 수, 업데이트 수, 총 시도 수)
    """
    if forecasts_long.empty:
        print("저장할 데이터가 없습니다")
        return 0, 0, 0

    conn = pymysql.connect(
        host=db_info["host"], port=db_info["port"], user=db_info["user"],
        password=db_info["password"], db=db_info["database"],
        charset="utf8mb4", autocommit=False,
    )
    try:
        cursor = conn.cursor()
        cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id INT AUTO_INCREMENT PRIMARY KEY,
            date DATE NOT NULL COMMENT '예측 대상 날짜',
            ticker VARCHAR(20) NOT NULL,
            indicator VARCHAR(50) NOT NULL,
            value DOUBLE,
            created_at DATE NOT NULL COMMENT '예측 실행 날짜 (DATE만)',
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
            UNIQUE KEY unique_date_ticker_indicator_created (date, ticker, indicator, created_at),
            INDEX idx_ticker (ticker),
            INDEX idx_date (date),
            INDEX idx_indicator (indicator),
            INDEX idx_created_at (created_at)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
        COMMENT='한국 기업 매출 예측 결과 (날짜별 버전 관리)'
        """)

        today_date = datetime.now().date()
        insert_sql = f"""
        INSERT INTO {table_name} (date, ticker, indicator, value, created_at)
        VALUES (%s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE value = VALUES(value), updated_at = CURRENT_TIMESTAMP
        """

        total_new = total_updated = total_attempted = 0
        total_rows = len(forecasts_long)

        for i in range(0, total_rows, batch_size):
            batch_df = forecasts_long.iloc[i:i + batch_size]
            batch_data = [
                (row['date'], row['ticker'], row['indicator'],
                 float(row['value']) if pd.notna(row['value']) else None, today_date)
                for _, row in batch_df.iterrows()
            ]
            cursor.executemany(insert_sql, batch_data)
            affected = cursor.rowcount

            if affected == len(batch_data):
                total_new += len(batch_data)
            elif affected == len(batch_data) * 2:
                total_updated += len(batch_data)
            else:
                updated = max(0, affected - len(batch_data))
                total_new += len(batch_data) - updated
                total_updated += updated
            total_attempted += len(batch_data)
            conn.commit()

        print(f"  저장 완료 | 시도 {total_attempted:,} · 신규 {total_new:,} · 갱신 {total_updated:,} · 실행일 {today_date}")
        return total_new, total_updated, total_attempted

    except Exception as e:
        conn.rollback()
        print(f"DB 저장 실패: {e}")
        raise
    finally:
        conn.close()

## 6. 일괄 예측 실행 함수 — `process_all_tickers()`

예측 결과를 `BATCH_SIZE`개씩 버퍼에 모았다가 Long 변환 후 한 번에 저장합니다.
`GC_EVERY`개마다 `gc.collect()`로 모델 적합 잔여 메모리를 정리합니다.

In [11]:
def process_all_tickers(
        tickers: List[str],
        db_info: dict,
        forecast_quarters: int = 8,
        min_data_periods: int = 24,
        batch_size: int = 50,
        gc_every: int = 10,
        verbose: bool = True
) -> Dict:
    """
    다수 ticker 예측 + 배치 저장.
    - batch_size: 버퍼에 모았다 한 번에 저장할 ticker 수
    - gc_every: 몇 ticker마다 gc.collect()를 호출할지
    """
    total = len(tickers)
    success_count = fail_count = 0
    failed_tickers = []
    batch_buffer = []

    print("=" * 80)
    print(f"예측 시작 | 대상 {total}개 · 예측 {forecast_quarters}분기 · 최소 {min_data_periods}분기 · 배치 {batch_size}")
    print("=" * 80)
    start_time = datetime.now()

    def _flush(buffer):
        if not buffer:
            return 0
        combined_long = convert_to_long_format(pd.concat(buffer, ignore_index=True))
        new, _, _ = save_forecasts_batch(combined_long, db_info, batch_size=batch_size)
        return new

    for i, ticker in enumerate(tqdm(tickers, desc="예측 진행"), 1):
        success, result_df, error_msg = forecast_quarterly_revenue(
            ticker=ticker, db_info=db_info,
            forecast_quarters=forecast_quarters,
            min_data_periods=min_data_periods, verbose=False,
        )
        if success:
            success_count += 1
            batch_buffer.append(result_df)
            if len(batch_buffer) >= batch_size:
                _flush(batch_buffer)
                batch_buffer = []
        else:
            fail_count += 1
            failed_tickers.append((ticker, error_msg))

        # 주기적 메모리 정리
        if gc_every and (i % gc_every == 0):
            gc.collect()

    # 남은 버퍼 저장
    _flush(batch_buffer)
    gc.collect()

    elapsed = (datetime.now() - start_time).total_seconds()

    print("\n" + "=" * 80)
    print("예측 완료")
    print("=" * 80)
    print(f"총 처리 시간: {elapsed:.1f}초 ({elapsed/60:.1f}분)")
    print(f"성공: {success_count}개 ({success_count/total*100:.1f}%)")
    print(f"실패: {fail_count}개 ({fail_count/total*100:.1f}%)")
    if total:
        print(f"평균 처리 시간: {elapsed/total:.2f}초/ticker")

    if failed_tickers:
        error_summary = {}
        for _, error in failed_tickers:
            error_summary[error] = error_summary.get(error, 0) + 1
        print(f"\n실패 원인 요약:")
        for error, count in sorted(error_summary.items(), key=lambda x: -x[1]):
            print(f"  {error}: {count}개")
        sample = failed_tickers if len(failed_tickers) <= 20 else failed_tickers[:20]
        print(f"\n실패 ticker {'목록' if len(failed_tickers) <= 20 else '샘플(20개)'}:")
        for t, e in sample:
            print(f"  {t}: {e}")

    return {'total': total, 'success': success_count, 'fail': fail_count,
            'failed_tickers': failed_tickers, 'elapsed_time': elapsed}

## 7. 실행

### 7-1. 공통 설정 · DB 연결

In [12]:
# ===== 예측 설정 =====
FORECAST_QUARTERS = 8     # 예측할 분기 수
MIN_DATA_PERIODS  = 24    # 최소 데이터 기간 (24분기 = 6년)
BATCH_SIZE        = 50    # 배치 저장 크기 (메모리 관리 판단에 따른 기본값)
GC_EVERY          = 10    # 몇 ticker마다 gc.collect()

db_info = get_db_info()
print("DB 연결 정보 준비 완료")

DB 연결 정보 준비 완료


### 7-2. 🔍 개별 종목 점검 (DB 저장 없음)

예측 타당성을 확인하려는 종목 코드를 넣고 실행하세요. 입력 매출 `tail(20)`과 모델별 예측값이 표시됩니다.

In [15]:
# 점검할 종목 코드 (예: 삼성전자)
INSPECT_TICKER = "278470"

result = inspect_ticker(
    ticker=INSPECT_TICKER,
    db_info=db_info,
    forecast_quarters=FORECAST_QUARTERS,
    tail_n=20,
)

# 반환 객체로 추가 분석 가능
# result["input_tail"]   # 입력 매출 최근 20분기
# result["forecast"]     # 모델별 예측값 (wide)

[메모리] forecast_sarima 실행 전: 437.04 MB
[메모리] find_best_sarima_params 실행 전: 437.04 MB
[메모리] find_best_sarima_params 실행 후: 437.09 MB (변화: +0.05 MB)
[메모리] forecast_sarima 실행 후: 437.09 MB (변화: +0.05 MB)
[메모리] forecast_ets 실행 전: 437.09 MB
[메모리] forecast_ets 실행 후: 437.09 MB (변화: +0.00 MB)
[메모리] forecast_theta 실행 전: 437.09 MB
[메모리] forecast_theta 실행 후: 437.09 MB (변화: +0.00 MB)
 종목: A278470   |   총 관측 분기: 33개   |   예측 분기: 8개

[ 예측 입력 매출 — 최근 20개 분기 (tail) ]
        revenue(천원)
period             
2021Q2   56,332,764
2021Q3   60,557,916
2021Q4   80,340,978
2022Q1   76,342,730
2022Q2   97,942,516
2022Q3   95,300,670
2022Q4  128,112,203
2023Q1  122,178,928
2023Q2  127,672,288
2023Q3  121,938,746
2023Q4  152,019,403
2024Q1  148,928,031
2024Q2  155,493,932
2024Q3  174,117,291
2024Q4  244,214,608
2025Q1  266,032,726
2025Q2  327,734,895
2025Q3  385,942,757
2025Q4  547,634,595
2026Q1  593,355,596

[ 모델별 예측값 ]
              SARIMA           ETS       Theta      Ensemble
2026Q2   684,583,363   638,649,59

### 7-3. 일괄 예측 + DB 저장

아래 셀에서 대상 종목을 정한 뒤 실행합니다.
- **테스트:** `RUN_TICKERS = all_tickers[:30]`
- **특정 종목만:** `RUN_TICKERS = ["005930", "000660"]`
- **구간 지정:** `all_tickers[TICKER_START:TICKER_END]`
- **전체:** `RUN_TICKERS = all_tickers`

In [17]:
# 전체 ticker 목록 조회
all_tickers = get_all_tickers(db_info)

# ----- 실행 대상 선택 (원하는 줄의 주석을 해제) -----
# RUN_TICKERS = all_tickers[:30]        # 테스트: 앞 30개
# RUN_TICKERS = ["005930", "000660"]  # 특정 종목만
# TICKER_START, TICKER_END = 0, 500
# RUN_TICKERS = all_tickers[TICKER_START:TICKER_END]   # 구간 지정
RUN_TICKERS = all_tickers           # 전체

print(f"실행 대상: {len(RUN_TICKERS)}개  (샘플: {RUN_TICKERS[:5]})")

매출 데이터 보유 ticker 1582개 조회 완료 (소스: korea_fs_data_from_DG)

실행 대상: 1582개  (샘플: ['A000020', 'A000040', 'A000050', 'A000070', 'A000080'])


In [18]:
result = process_all_tickers(
    tickers=RUN_TICKERS,
    db_info=db_info,
    forecast_quarters=FORECAST_QUARTERS,
    min_data_periods=MIN_DATA_PERIODS,
    batch_size=BATCH_SIZE,
    gc_every=GC_EVERY,
    verbose=True,
)

예측 시작 | 대상 1582개 · 예측 8분기 · 최소 24분기 · 배치 50


예측 진행:   0%|          | 0/1582 [00:00<?, ?it/s]

[메모리] forecast_sarima 실행 전: 440.60 MB
[메모리] find_best_sarima_params 실행 전: 440.60 MB
[메모리] find_best_sarima_params 실행 후: 442.54 MB (변화: +1.95 MB)
[메모리] forecast_sarima 실행 후: 442.54 MB (변화: +1.95 MB)
[메모리] forecast_ets 실행 전: 442.54 MB
[메모리] forecast_ets 실행 후: 442.55 MB (변화: +0.00 MB)
[메모리] forecast_theta 실행 전: 442.55 MB
[메모리] forecast_theta 실행 후: 442.55 MB (변화: +0.00 MB)
[메모리] forecast_sarima 실행 전: 442.79 MB
[메모리] find_best_sarima_params 실행 전: 442.79 MB
[메모리] find_best_sarima_params 실행 후: 442.81 MB (변화: +0.03 MB)
[메모리] forecast_sarima 실행 후: 442.81 MB (변화: +0.03 MB)
[메모리] forecast_ets 실행 전: 442.81 MB
[메모리] forecast_ets 실행 후: 442.81 MB (변화: +0.00 MB)
[메모리] forecast_theta 실행 전: 442.81 MB
[메모리] forecast_theta 실행 후: 442.82 MB (변화: +0.00 MB)
[메모리] forecast_sarima 실행 전: 442.83 MB
[메모리] find_best_sarima_params 실행 전: 442.83 MB
[메모리] find_best_sarima_params 실행 후: 442.82 MB (변화: -0.01 MB)
[메모리] forecast_sarima 실행 후: 442.82 MB (변화: -0.01 MB)
[메모리] forecast_ets 실행 전: 442.82 MB
[메모리] forecast_ets 실행 후

In [17]:
pd.DataFrame(result)


,total,success,fail,failed_tickers,elapsed_time
0,30,28,2,"(A0001A0, 데이터 부족 (4개 < 24개))",172.724311
1,30,28,2,"(A0004V0, 데이터 부족 (3개 < 24개))",172.724311


### 7-4. 실패 종목 재시도 (선택)

In [ ]:
if result['failed_tickers']:
    retry_tickers = [t[0] for t in result['failed_tickers']]
    print(f"재시도 대상: {len(retry_tickers)}개\n")
    retry_result = process_all_tickers(
        tickers=retry_tickers,
        db_info=db_info,
        forecast_quarters=FORECAST_QUARTERS,
        min_data_periods=MIN_DATA_PERIODS,
        batch_size=BATCH_SIZE,
        gc_every=GC_EVERY,
        verbose=True,
    )
else:
    print("실패한 ticker가 없습니다.")